# 04_capas_terreno

In [ ]:
# ---- 0. Packages ----
import json, math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window, from_bounds
from scipy import ndimage

SEP = "=" * 40

In [ ]:
# ---- 1. Configuration ----
STACKS     = ["A_T148", "D_T011", "D_T112", "A_T047"]
FECHA_MIN  = "2026-06-01"
EPSG, RES = 32618, 20
X0_MALLA, Y1_MALLA = 272_880, 796_440
NY_MALLA, NX_MALLA = 9144, 3351
PISO = 1e-3
UMBRAL_AGUA_REF = -15.0
URL_DEM = "https://copernicus-dem-30m.s3.amazonaws.com/{n}/{n}.tif"
URL_WC = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{t}_Map.tif"

# ---- Folders ----
DIR_BASE = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "metadatos").exists()), Path(".."))
DIR_RECORTES = DIR_BASE / "recortes"
DIR_AUX = DIR_BASE / "entrenamiento" / "insumos" / "terreno"
DIR_AUX.mkdir(parents=True, exist_ok=True)
RUTA_FUENTES = DIR_AUX / "fuentes.json"
FUENTES = json.loads(RUTA_FUENTES.read_text(encoding="utf-8")) if RUTA_FUENTES.exists() else {}


def renglon(nombre, valor, fmt=""):
    print(f"{nombre:<28}: {valor:{fmt}}")


print(SEP)
renglon("Carpeta del proyecto", DIR_BASE)
renglon("Salida", DIR_AUX)

In [ ]:
# ---- 2. Common functions ----
def a_db(x):
    return 10 * np.log10(np.clip(x, PISO, None))


def fechas_stack(stack):
    filas = []
    for ruta in sorted((DIR_RECORTES / stack).glob("*/*.npz")):
        f = ruta.parent.name
        if f >= FECHA_MIN:
            filas.append({"fecha": f, "ruta": ruta, "frames": ruta.stem.count("F0")})
    inv = pd.DataFrame(filas)
    if inv.empty:
        return inv
    return inv.sort_values(["fecha", "frames"], ascending=[True, False]).drop_duplicates("fecha").reset_index(drop=True)


class Malla:
    def __init__(self, r0, r1, c0, c1):
        self.r0, self.r1, self.c0, self.c1 = r0, r1, c0, c1
        self.forma = (r1 - r0, c1 - c0)
        self.x0, self.y1 = X0_MALLA + c0 * RES, Y1_MALLA - r0 * RES
        self.transf = from_origin(self.x0, self.y1, RES, RES)
        self.caja = (self.x0, self.y1 - self.forma[0] * RES, self.x0 + self.forma[1] * RES, self.y1)

    def lonlat(self):
        from rasterio.warp import transform_bounds
        return transform_bounds(f"EPSG:{EPSG}", "EPSG:4326", *self.caja, densify_pts=21)


MALLA_ZONA = Malla(0, NY_MALLA, 0, NX_MALLA)


def guardar_aux(nombre, a, malla, dtype="float32", nodata=np.nan, fuente=None):
    ruta = DIR_AUX / f"{nombre}.tif"
    perfil = {"driver": "GTiff", "height": malla.forma[0], "width": malla.forma[1], "count": 1, "dtype": dtype,
              "crs": f"EPSG:{EPSG}", "transform": malla.transf, "nodata": nodata, "compress": "deflate",
              "tiled": True}
    with rasterio.open(ruta, "w", **perfil) as dst:
        dst.write(a.astype(dtype), 1)
    if fuente:
        FUENTES[nombre] = {**fuente, "bytes": ruta.stat().st_size,
                           "guardado": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        RUTA_FUENTES.write_text(json.dumps(FUENTES, ensure_ascii=False, indent=2), encoding="utf-8")
    return a


def leer_aux(nombre, malla):
    ruta = DIR_AUX / f"{nombre}.tif"
    if not ruta.exists():
        return None
    with rasterio.open(ruta) as src:
        assert src.shape == malla.forma and src.transform == malla.transf, f"{ruta.name} está en otra malla"
        return src.read(1)


def a_malla(ruta, malla, resampling, dtype=np.float32, nodata=np.nan, destino=None):
    out = destino if destino is not None else np.full(malla.forma, nodata, dtype)
    fuente = f"/vsicurl/{ruta}" if str(ruta).startswith("http") else str(ruta)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
                      GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="2"):
        with rasterio.open(fuente) as src:
            b = transform_bounds(f"EPSG:{EPSG}", src.crs, *malla.caja, densify_pts=21)
            w = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            w = Window(w.col_off - 3, w.row_off - 3, w.width + 6, w.height + 6)
            try:
                w = w.intersection(Window(0, 0, src.width, src.height))
            except rasterio.errors.WindowError:
                return out
            a = src.read(1, window=w)
            tmp = np.full(malla.forma, nodata, dtype)
            reproject(a, tmp, src_transform=src.window_transform(w), src_crs=src.crs, src_nodata=src.nodata,
                      dst_transform=malla.transf, dst_crs=f"EPSG:{EPSG}", dst_nodata=nodata, resampling=resampling)
    vacio = np.isnan(out) if np.issubdtype(np.dtype(dtype), np.floating) else (out == nodata)
    out[vacio] = tmp[vacio]
    return out


def capas_fijas_zona(agua_nisar_zona):
    m = MALLA_ZONA
    dem, wc = leer_aux("DEM", m), leer_aux("WORLDCOVER", m)
    if dem is None:
        o, s_, e, n = m.lonlat()
        nombres = [f"Copernicus_DSM_COG_10_{'N' if la >= 0 else 'S'}{abs(la):02d}_00_{'W' if lo < 0 else 'E'}{abs(lo):03d}_00_DEM"
                   for la in range(math.floor(s_), math.floor(n) + 1) for lo in range(math.floor(o), math.floor(e) + 1)]
        dem = np.full(m.forma, np.nan, np.float32)
        for nom in nombres:
            a_malla(URL_DEM.format(n=nom), m, Resampling.bilinear, destino=dem)
            renglon("DEM", nom)
        guardar_aux("DEM", dem, m, fuente={"capa": "Copernicus DEM GLO-30 (DSM)", "teselas": nombres})
    if wc is None:
        o, s_, e, n = m.lonlat()
        teselas = sorted({f"{'N' if la >= 0 else 'S'}{abs(la):02d}{'W' if lo < 0 else 'E'}{abs(lo):03d}"
                          for la in range(math.floor(s_ / 3) * 3, math.floor(n / 3) * 3 + 1, 3)
                          for lo in range(math.floor(o / 3) * 3, math.floor(e / 3) * 3 + 1, 3)})
        wc = np.zeros(m.forma, np.uint8)
        for t in teselas:
            a_malla(URL_WC.format(t=t), m, Resampling.mode, dtype=np.uint8, nodata=0, destino=wc)
            renglon("WorldCover", t)
        guardar_aux("WORLDCOVER", wc, m, "uint8", 0, {"capa": "ESA WorldCover 2021 v200", "teselas": teselas})
    pend, hand = leer_aux("PEND", m), leer_aux("HAND", m)
    if pend is None:
        d = ndimage.gaussian_filter(np.nan_to_num(dem, nan=float(np.nanmedian(dem))), 1.5)
        gy, gx = np.gradient(d, RES)
        pend = guardar_aux("PEND", np.degrees(np.arctan(np.hypot(gx, gy))).astype(np.float32), m,
                           fuente={"capa": "pendiente de Copernicus DEM (suavizado σ=1,5 px)"})
    if hand is None:
        agua = ((wc == 80) | agua_nisar_zona) & np.isfinite(dem)
        _, (fi, ci) = ndimage.distance_transform_edt(~agua, return_indices=True)
        hand = guardar_aux("HAND", (dem - dem[fi, ci]).astype(np.float32), m,
                           fuente={"capa": "altura sobre el agua más cercana (WorldCover 80 ∪ agua NISAR)"})
    return {"DEM": dem, "WORLDCOVER": wc, "PEND": pend, "HAND": hand}

In [ ]:
# ---- 3. NISAR water and terrain layers ----
agua_z = np.zeros((NY_MALLA, NX_MALLA), bool)
for st in STACKS:
    inv = fechas_stack(st)
    if inv.empty:
        continue
    with np.load(inv["ruta"].iloc[0]) as z:
        hh = z["HH"]
        agua_z |= np.nan_to_num(a_db(ndimage.uniform_filter(np.nan_to_num(hh, nan=1.0), 5)), nan=0) < UMBRAL_AGUA_REF
agua_z = ndimage.binary_opening(agua_z, np.ones((3, 3)))
CAPAS = capas_fijas_zona(agua_z)
del agua_z

print(SEP)
for nombre in CAPAS:
    renglon(nombre, DIR_AUX / f"{nombre}.tif")
print(SEP)